In [2]:
import pandas as pd

df = pd.read_csv('skin_irritation_2Ddesc.csv')
print('shape:', df.shape)
df.head()

shape: (39, 220)


,Chemical_Name,standardized_smi,label,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,Heptanal,CCCCCCC=O,1,9.768009,9.768009,0.750000,0.750000,0.395123,9.125000,114.188,...,0,0,0,0,0,0,0,0,4,0
1,4-Methylthio benzaldehyde,CSc1ccc(C=O)cc1,0,10.199308,10.199308,0.734074,0.734074,0.477177,9.300000,152.218,...,1,0,0,0,0,0,0,0,0,0
2,Heptyl butyrate,CCCCCCCOC(=O)CCC,0,10.909695,10.909695,0.043526,-0.043526,0.429221,10.000000,186.295,...,0,0,0,0,0,0,0,0,4,0
3,Hydroxycitronellal,COC(=O)c1ccccc1N=CCC(C)CCCC(C)(C)O,0,11.646816,11.646816,0.369467,-0.590016,0.579721,13.318182,305.418,...,0,0,0,0,0,0,0,0,0,0
4,Methyl caproate,CCCCCC(=O)OC,0,10.464402,10.464402,0.094028,-0.094028,0.427720,9.111111,130.187,...,0,0,0,0,0,0,0,0,2,0


In [3]:
y = df['label']
X = df.drop(columns=['Chemical_Name', 'standardized_smi', 'label'])

X = X.dropna(axis=1)                 # NaN 열 제거
print('NaN 제거:', X.shape)

X = X.loc[:, X.std() >= 0.01]        # 저분산 제거
print('std 필터:', X.shape)

NaN 제거: (39, 209)
std 필터: (39, 144)


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

'''
descriptor 단위가 달라서 표준화 (사실 2D descriptor 사용할 때는 normalization이 필수!)
https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html

standardscaler는 descriptor 값을 평균과 표준편차로 표준화시키는 방법. (나는 고등학교 때 수학 시간에 확률과 통계 시간에 배웠던 것 같은데...)

또 다른 표준화 방식은 descriptor의 최대, 최소 값을 이용하는 것.
https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html
'''

X_scaled = StandardScaler().fit_transform(X)   # descriptor값 표준화

clf = LogisticRegression(max_iter=5000)
clf.fit(X_scaled, y)
print('train :', clf.score(X_scaled, y))
print('CV5   :', cross_val_score(clf, X_scaled, y, cv=5).mean().round(3))

train : 0.9743589743589743
CV5   : 0.789


In [5]:
from sklearn.feature_selection import SelectKBest, f_classif

selector = SelectKBest(f_classif, k=10)
selector.fit(X, y)
print(list(X.columns[selector.get_support()]))


['MinAbsEStateIndex', 'BertzCT', 'Chi0', 'Chi1', 'PEOE_VSA7', 'SlogP_VSA6', 'HeavyAtomCount', 'MolMR', 'fr_C_O_noCOO', 'fr_ester']


In [6]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

k_candidates = [5, 10, 15, 20, 25, 30, 40, 50]
best_k, best_score = None, -1

# TODO: 각 k에 대해 SelectKBest → StandardScaler → LogReg CV5
#       k, acc 출력 + best_k/best_score 업데이트
#       예시 출력: 'k=10: 0.793'


In [7]:
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# TODO: best_k로 SelectKBest → StandardScaler → MLPClassifier(random_state=0, max_iter=2000)
#       CV5 점수 출력 + 실습 1 결과와 한 줄 비교


In [8]:
from sklearn.svm import SVC
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# TODO: best_k로 X_sel 만들고, C × kernel 조합 CV5 출력
#       예시 출력: 'C=0.1, kernel=linear : 0.775'

In [9]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

k_candidates = [5, 10, 15, 20, 25, 30, 40, 50]

best_k, best_score = None, -1

for k in k_candidates:
    selector = SelectKBest(f_classif, k=k)
    X_sel = selector.fit_transform(X, y)

    X_scaled = StandardScaler().fit_transform(X_sel)

    clf = LogisticRegression(max_iter=5000)
    score = cross_val_score(clf, X_scaled, y, cv=5).mean()

    print(f'k={k}: {score:.3f}')

    if score > best_score:
        best_score = score
        best_k = k

print(f'\nBest k: {best_k}, Best CV5: {best_score:.3f}')

k=5: 0.668
k=10: 0.793
k=15: 0.793
k=20: 0.793
k=25: 0.743
k=30: 0.743
k=40: 0.664
k=50: 0.661

Best k: 10, Best CV5: 0.793


In [10]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

selector = SelectKBest(f_classif, k=best_k)
X_sel = selector.fit_transform(X, y)

X_scaled = StandardScaler().fit_transform(X_sel)

logreg = LogisticRegression(max_iter=5000)
mlp = MLPClassifier(random_state=0, max_iter=2000)

log_score = cross_val_score(logreg, X_scaled, y, cv=5).mean()
mlp_score = cross_val_score(mlp, X_scaled, y, cv=5).mean()

print(f'Logistic Regression CV5: {log_score:.3f}')
print(f'MLP CV5: {mlp_score:.3f}')

if mlp_score > log_score:
    print('MLP가 로지스틱보다 더 좋음')
else:
    print('MLP가 로지스틱보다 좋지 않거나 비슷함')

Logistic Regression CV5: 0.793
MLP CV5: 0.764
MLP가 로지스틱보다 좋지 않거나 비슷함


In [11]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC

selector = SelectKBest(f_classif, k=best_k)
X_sel = selector.fit_transform(X, y)

X_scaled = StandardScaler().fit_transform(X_sel)

C_list = [0.1, 1, 10, 100]
kernel_list = ['linear', 'rbf', 'poly']

best_C, best_kernel, best_score = None, None, -1

for C in C_list:
    for kernel in kernel_list:
        svm = SVC(C=C, kernel=kernel)
        score = cross_val_score(svm, X_scaled, y, cv=5).mean()

        print(f'C={C}, kernel={kernel}: {score:.3f}')

        if score > best_score:
            best_score = score
            best_C = C
            best_kernel = kernel

print(f'\nBest SVM: C={best_C}, kernel={best_kernel}, CV5={best_score:.3f}')

C=0.1, kernel=linear: 0.775
C=0.1, kernel=rbf: 0.668
C=0.1, kernel=poly: 0.668
C=1, kernel=linear: 0.689
C=1, kernel=rbf: 0.746
C=1, kernel=poly: 0.693
C=10, kernel=linear: 0.711
C=10, kernel=rbf: 0.768
C=10, kernel=poly: 0.614
C=100, kernel=linear: 0.711
C=100, kernel=rbf: 0.689
C=100, kernel=poly: 0.664

Best SVM: C=0.1, kernel=linear, CV5=0.775
